# GISPR Module 6 — Raster Operations in Python and R

**Course:** GIS Spatial Analysis with Python and R (GISPR)
**Module:** 6 — Raster Operations: Map Algebra, Zonal Statistics, Raster–Vector Integration, and xarray
**Builds on:** Module 5 (raster data model, `inspect_raster()`, NoData) and Modules 3–4 (the WRIA 8 salmon watersheds)

---

### Module Task

> *For each of our 15 watersheds we'll calculate: mean elevation, mean slope, percent forest, and how much of it is gentle, low-elevation land that isn't open water — the first screen for floodplain and off-channel restoration sites.*

In ArcGIS Pro that's **Extract by Mask → Slope → Raster Calculator → Zonal Statistics as Table → Tabulate Area → Join Field**, repeated by hand every time the data changes. Tonight you script the same chain with **ArcPy first**, then with the open-source raster stack (**rasterio / NumPy / rasterstats** in Python, **terra / exactextractr** in R), on the real WRIA 8 DEM and NLCD 2021 rasters you downloaded for Module 5 and the watersheds from Modules 3–4.

### What you'll accomplish

| # | Learning outcome | Section |
|---|---|---|
| 1 | Run raster algebra, reclassification and format conversion with `arcpy.sa`, and pass rasters between ArcPy and NumPy | 2 |
| 2 | Warp rasters: clip to a boundary, reproject, and resample with the right method for the data type | 3 |
| 3 | Apply local, focal, and global map algebra | 4 |
| 4 | Calculate zonal statistics with ArcPy, the R-ArcGIS Bridge, `rasterstats`, and `exactextractr` | 5–6 |
| 5 | Rasterize vector zones and vectorize class rasters | 7 |
| 6 | Slice and summarize a multidimensional (NetCDF) time series with `xarray` and `stars` | 8 |

The **Guided Lab** (Section 9) chains it all into the recovery team's table — mostly copy, run, and change one thing at a time.

### Kernel reminder
- 🟢 **`[Python]`** cells — Python 3 kernel (JupyterHub) or your cloned ArcGIS Pro environment
- 🔵 **`[R]`** cells — switch to the **R** kernel
- 🟠 **`[Python / ArcPy]`** cells — 🏠 **LOCAL ONLY**: need ArcGIS Pro + Spatial Analyst. Elsewhere they print a note and skip, so the notebook still runs top to bottom.

Run the cells **in order within each language** — later cells reuse variables and files made earlier.

---
## Section 1 — Setup: the Module 5 Rasters and the Module 3 Watersheds

| Input | File | CRS | Notes |
|---|---|---|---|
| Elevation | `module5_rasters/wria8_dem_30m.tif` | EPSG:32610 (UTM 10N, metres) | USGS 3DEP, 30 m, float32, covers all of WRIA 8 |
| Land cover | `module5_rasters/wria8_nlcd2021_30m.tif` | EPSG:32610 | NLCD 2021 classes, uint8, same grid as the DEM |
| Watersheds | `salmon_watersheds.gdb` → `watersheds` | EPSG:2926 (WA State Plane N, US ft) | The 15 HUC-12s from Modules 3–4 |
| Study area | `salmon_watersheds.gdb` → `wria8_boundary` | EPSG:2926 | Used to clip the rasters |

**Where the files go:** keep `module5_rasters.zip` next to this notebook (it's in the course GitHub repo and on Canvas) — the setup cell unzips it the first time. Point `DATA` at the same `GISPR_Module3_Data/data` folder you used in Module 4. Outputs go to `outputs/module6/`.

**NOTE!!**
> 📋 **[Terminal]** if `rasterstats` is missing (it often is in a cloned ArcGIS Pro env): `conda install rasterstats -c conda-forge`

🟢 **`[Python]`**

In [ ]:
# [Python] Section 1 — Setup: find the Module 3 vectors and the Module 5 rasters
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

# ── 1. Module 3 data folder (salmon_watersheds.gdb) — same folder Module 4 used ──
DATA = Path("data")                       # ← edit if your GISPR_Module3_Data/data lives elsewhere
for cand in [DATA, Path("GISPR_Module3_Data/data"), Path("../GISPR_Module3_Data/data"),
             Path("../Module 3/GISPR_Module3_Data/data")]:
    if (cand / "salmon_watersheds.gdb").exists():
        DATA = cand
        break
WS_GDB = DATA / "salmon_watersheds.gdb"
assert WS_GDB.exists(), "Can't find salmon_watersheds.gdb — set DATA to your GISPR_Module3_Data/data folder"

# ── 2. Module 5 rasters — unzip module5_rasters.zip the first time ────────────
RAST = Path("module5_rasters")
DEM_TIF, NLCD_TIF = RAST / "wria8_dem_30m.tif", RAST / "wria8_nlcd2021_30m.tif"
if not DEM_TIF.exists():
    zips = [z for z in [Path("module5_rasters.zip"), DATA / "module5_rasters.zip",
                        Path("../Module 5/module5_rasters.zip")] if z.exists()]
    assert zips, "Put module5_rasters.zip (course GitHub repo / Canvas) next to this notebook"
    zipfile.ZipFile(zips[0]).extractall(RAST)
    print(f"Unzipped {zips[0]} → {RAST}/")

OUT = Path("outputs/module6")
OUT.mkdir(parents=True, exist_ok=True)
RASTER_EPSG = 32610      # the rasters: UTM zone 10N, metres
COURSE_EPSG = 2926       # the Module 3–4 vectors: WA State Plane North, US survey feet

# ── 3. Vector layers from Module 3 (lower-case column names, as in the Module 3 lab) ──
watersheds = gpd.read_file(WS_GDB, layer="watersheds").rename(columns=str.lower)
wria8      = gpd.read_file(WS_GDB, layer="wria8_boundary").rename(columns=str.lower)

for name, path in [("DEM", DEM_TIF), ("NLCD", NLCD_TIF)]:
    with rasterio.open(path) as src:
        print(f"{name:<11} EPSG:{src.crs.to_epsg()}   {src.res[0]:.0f} m cells   {src.width} × {src.height}")
print(f"{'watersheds':<11} EPSG:{watersheds.crs.to_epsg()}   units: {watersheds.crs.axis_info[0].unit_name}   "
      f"{len(watersheds)} polygons")

**Decide on coordinate reference system (CRS).** The rasters are in UTM metres; the watersheds are in State Plane feet. You can't do raster–vector work until they match, and there are two ways to get there:

- **Warp the rasters** to EPSG:2926 — resamples 6.9 million cells and slightly degrades the values.
- **Reproject the vectors** to EPSG:32610 — moves 15 polygon outlines, exactly.

Reprojecting the vectors is cheaper and lossless, so that's tonight's default. (Section 3 shows how to warp a raster when you *do* need to — e.g., to deliver it on the State Plane grid.)

In [ ]:
# [Python] Bring the vectors to the rasters' CRS, then run the Module 5 checklist
ws   = watersheds.to_crs(RASTER_EPSG)      # 15 watersheds, now in UTM metres
wria = wria8.to_crs(RASTER_EPSG)           # WRIA 8 outline, for clipping

def inspect_raster(path, band=1):
    """Module 5 first-inspection checklist — run before ANY raster operation."""
    with rasterio.open(path) as src:
        arr = src.read(band, masked=True)
        info = dict(epsg=src.crs.to_epsg(), res=src.res[0], unit={"metre": "m", "US survey foot": "ft"}.get(src.crs.linear_units, src.crs.linear_units), rows=src.height, cols=src.width,
                    dtype=src.dtypes[band - 1], nodata=src.nodata, n_nodata=int(arr.mask.sum()),
                    vmin=float(arr.min()), vmax=float(arr.max()))
    print(f"── {Path(path).name}")
    print(f"   EPSG:{info['epsg']}  |  {info['res']:g} {info['unit']} cells  |  {info['rows']:,} rows × {info['cols']:,} cols")
    print(f"   dtype {info['dtype']}  |  NoData value {info['nodata']}  ({info['n_nodata']:,} NoData cells)")
    print(f"   values {info['vmin']:.1f} to {info['vmax']:.1f}")
    return info

dem_info  = inspect_raster(DEM_TIF)
nlcd_info = inspect_raster(NLCD_TIF)

with rasterio.open(NLCD_TIF) as src:
    codes = np.unique(src.read(1))
print("\nNLCD classes present:", codes.tolist())
print("Watersheds now in EPSG:", ws.crs.to_epsg())

🔵 **`[R]`**

In [ ]:
# [R] Section 1 — Setup with terra + sf: same folders, same checks
library(terra)
library(sf)

DATA <- "data"                                   # ← edit if your GISPR_Module3_Data/data lives elsewhere
for (cand in c(DATA, "GISPR_Module3_Data/data", "../GISPR_Module3_Data/data",
               "../Module 3/GISPR_Module3_Data/data")) {
  if (dir.exists(file.path(cand, "salmon_watersheds.gdb"))) { DATA <- cand; break }
}
WS_GDB <- file.path(DATA, "salmon_watersheds.gdb")
stopifnot("Set DATA to your GISPR_Module3_Data/data folder" = dir.exists(WS_GDB))

RAST     <- "module5_rasters"
DEM_TIF  <- file.path(RAST, "wria8_dem_30m.tif")
NLCD_TIF <- file.path(RAST, "wria8_nlcd2021_30m.tif")
if (!file.exists(DEM_TIF)) {
  zips <- Filter(file.exists, c("module5_rasters.zip", file.path(DATA, "module5_rasters.zip"),
                                "../Module 5/module5_rasters.zip"))
  stopifnot("Put module5_rasters.zip next to this notebook" = length(zips) > 0)
  unzip(zips[1], exdir = RAST)
}
OUT <- "outputs/module6"; dir.create(OUT, recursive = TRUE, showWarnings = FALSE)
RASTER_EPSG <- 32610

lower_names <- function(x) {                     # lower-case attribute names, leave the geometry column alone
  keep <- names(x) != attr(x, "sf_column"); names(x)[keep] <- tolower(names(x)[keep]); x
}
watersheds <- lower_names(st_read(WS_GDB, layer = "watersheds",     quiet = TRUE))
wria8      <- lower_names(st_read(WS_GDB, layer = "wria8_boundary", quiet = TRUE))

dem  <- rast(DEM_TIF)
nlcd <- rast(NLCD_TIF)
dem                                              # CRS, extent, resolution in one printout
cat("Watersheds EPSG:", st_crs(watersheds)$epsg, "→ reproject to", RASTER_EPSG, "\n")

ws   <- st_transform(watersheds, RASTER_EPSG)    # vectors to the rasters' CRS (cheap, exact)
wria <- st_transform(wria8, RASTER_EPSG)
head(freq(nlcd), 16)                             # cells per NLCD class — categorical sanity check

🟠 **`[Python / ArcPy]`**
We run this locally so we need to set up our paths and file checks again here separately

In [ ]:
from pathlib import Path
import zipfile

# ── 1. Module 3 data folder (salmon_watersheds.gdb) — same folder Module 4 used ──
DATA = Path("data")                       # ← edit if your GISPR_Module3_Data/data lives elsewhere
for cand in [DATA, Path("GISPR_Module3_Data/data"), Path("../GISPR_Module3_Data/data"),
             Path("../Module 3/GISPR_Module3_Data/data")]:
    if (cand / "salmon_watersheds.gdb").exists():
        DATA = cand
        break
WS_GDB = DATA / "salmon_watersheds.gdb"
assert WS_GDB.exists(), "Can't find salmon_watersheds.gdb — set DATA to your GISPR_Module3_Data/data folder"

# ── 2. Module 5 rasters — unzip module5_rasters.zip the first time ────────────
RAST = Path("module5_rasters")
DEM_TIF, NLCD_TIF = RAST / "wria8_dem_30m.tif", RAST / "wria8_nlcd2021_30m.tif"
if not DEM_TIF.exists():
    zips = [z for z in [Path("module5_rasters.zip"), DATA / "module5_rasters.zip",
                        Path("../Module 5/module5_rasters.zip")] if z.exists()]
    assert zips, "Put module5_rasters.zip (course GitHub repo / Canvas) next to this notebook"
    zipfile.ZipFile(zips[0]).extractall(RAST)
    print(f"Unzipped {zips[0]} → {RAST}/")

OUT = Path("outputs/module6")
OUT.mkdir(parents=True, exist_ok=True)
RASTER_EPSG = 32610      # the rasters: UTM zone 10N, metres
COURSE_EPSG = 2926 

In [ ]:
# [Python / ArcPy] ESRI setup — Spatial Analyst, a scratch geodatabase, and the same inputs
# 🏠 LOCAL ONLY: needs ArcGIS Pro + Spatial Analyst. On JupyterHub this prints a note and moves on.
try:
    import arcpy
    from arcpy.sa import *                       # Raster, Slope, Reclassify, Con, FocalStatistics, …
    if arcpy.CheckOutExtension("Spatial") != "CheckedOut":
        raise RuntimeError("Spatial Analyst license is not available")
    arcpy.env.overwriteOutput = True
    if not arcpy.Exists(str(OUT / "module6.gdb")):
        arcpy.management.CreateFileGDB(str(OUT), "module6.gdb")
    arcpy.env.workspace  = str((OUT / "module6.gdb").resolve())   # tool outputs land here
    DEM_A, NLCD_A = str(DEM_TIF.resolve()), str(NLCD_TIF.resolve())
    WS_FC   = str((WS_GDB / "watersheds").resolve())               # feature classes, read in place
    WRIA_FC = str((WS_GDB / "wria8_boundary").resolve())
    arcpy.env.snapRaster = DEM_A                 # every output lines up with the DEM grid
    HAS_ARCPY = True
    print("✅ arcpy ready — Spatial Analyst checked out; outputs →", arcpy.env.workspace)
except (ModuleNotFoundError, RuntimeError) as e:
    HAS_ARCPY = False
    print(f"ℹ️  arcpy not available here ({e}).")
    print("    ArcPy cells will skip. Run them in ArcGIS Pro (or your cloned Pro env) to see the ESRI path.")

In [ ]:
# [Python / ArcPy] The same checklist through arcpy: Raster properties + Describe
if HAS_ARCPY:
    dem_r = arcpy.Raster(DEM_A)
    if dem_r.minimum is None:                    # no statistics yet → calculate them once
        arcpy.management.CalculateStatistics(DEM_A)
        dem_r = arcpy.Raster(DEM_A)
    sr = dem_r.spatialReference
    print(f"DEM        {sr.name} (WKID {sr.factoryCode}) | {dem_r.meanCellWidth:g} {sr.linearUnitName} cells | "
          f"{dem_r.width} × {dem_r.height} | {dem_r.pixelType} | {dem_r.minimum:.1f} to {dem_r.maximum:.1f}")
    ws_sr = arcpy.Describe(WS_FC).spatialReference
    print(f"Watersheds {ws_sr.name} (WKID {ws_sr.factoryCode}) | linear unit: {ws_sr.linearUnitName}")
    print("ArcPy projects vectors on the fly in most tools — the open-source libraries do NOT. Keep that in mind.")
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

> 🔧 **Try it yourself:** Run `inspect_raster(NLCD_TIF)` again and compare it with the DEM: same CRS, same cell size, same rows × cols? That match is what lets you combine the two rasters cell by cell in Section 4. Then print `watersheds.crs` and `ws.crs` — which one would you hand to a raster tool?

---
## Section 2 — Raster Geoprocessing with ArcPy

`arcpy.sa` turns the Raster Calculator into Python. Rasters are **objects**: `+ - * /`, `<`, `&` and `|` work on whole grids, Spatial Analyst tools (`Slope`, `Reclassify`, `Con`, …) return new Raster objects, and nothing is written to disk until you call `.save()`.

Two patterns to take away:

1. **Tool chaining** — the expression you'd type into Raster Calculator is now one line you can loop over 50 rasters.
2. **The ArcPy ↔ NumPy bridge** — `arcpy.RasterToNumPyArray()` and `arcpy.NumPyArrayToRaster()` let you use ArcPy for licensed tools and I/O, and NumPy/SciPy for any custom math ArcGIS doesn't have a tool for. A GeoTIFF saved by ArcPy is just a file to `rasterio` and `terra`.

*ArcGIS API for Python note:* `arcgis.raster` runs the same kinds of operations **server-side** on imagery hosted in ArcGIS Online/Enterprise (it needs ArcGIS Image and consumes credits). The desktop `arcpy` path is what we use tonight.

🟠 **`[Python / ArcPy]`**

In [ ]:
# [Python / ArcPy] Raster algebra, reclassification, and format conversion with arcpy.sa
if HAS_ARCPY:
    dem_r = Raster(DEM_A)
    ARC_GDB = arcpy.env.scratchGDB
    arcpy.env.workspace = ARC_GDB
    # 1. Raster algebra — a Raster Calculator expression, written as Python
    slope_r    = Slope(dem_r, "PERCENT_RISE")                 # Spatial Analyst ▸ Slope
    suitable_r = (slope_r < 15) & (dem_r < 1000)              # 1 = gentle AND below 1000 m
    suitable_r.save("suitable_basic")                         # → module6.gdb/suitable_basic

    # 2. Reclassify — the Reclassify tool with a RemapRange table
    remap = RemapRange([[-10, 100, 1],                        # lowland / floodplain
                        [100, 500, 2],                        # foothill
                        [500, 1000, 3],                       # montane
                        [1000, 2000, 4]])                     # subalpine
    elev_class_r = Reclassify(dem_r, "Value", remap)
    elev_class_r.save("elev_class_arcpy")

    # 3. Format conversion — gdb raster → compact 8-bit GeoTIFF that rasterio / terra can open
    arcpy.management.CopyRaster("elev_class_arcpy", str((OUT / "elev_class_arcpy.tif").resolve()),
                                pixel_type="8_BIT_UNSIGNED")

    arcpy.management.BuildRasterAttributeTable("elev_class_arcpy", "Overwrite")
    with arcpy.da.SearchCursor("elev_class_arcpy", ["Value", "Count"]) as cursor:
        for value, count in cursor:
            print(f"  elevation class {value}: {count:>9,} cells")
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

In [ ]:
# [Python / ArcPy + NumPy] The bridge: ArcPy for licensed tools and I/O, NumPy/SciPy for custom math
if HAS_ARCPY:
    import numpy as np
    from scipy import ndimage

    slope_arr = arcpy.RasterToNumPyArray(slope_r, nodata_to_value=np.nan)     # Raster object → NumPy array
    print(f"Slope as an array: {slope_arr.shape}, median {np.nanmedian(slope_arr):.1f} % rise")

    # Custom focal metric with no single ArcGIS tool: share of steep (>30 %) cells in an 11×11 window (~330 m)
    steep_share = ndimage.uniform_filter((slope_arr > 30).astype("float32"), size=11)

    lower_left = arcpy.Point(slope_r.extent.XMin, slope_r.extent.YMin)      # array → Raster needs the
    steep_r = arcpy.NumPyArrayToRaster(steep_share, lower_left,              # lower-left corner and cell size
                                       slope_r.meanCellWidth, slope_r.meanCellHeight)
    steep_r.save("steep_share")
    arcpy.management.DefineProjection("steep_share", slope_r.spatialReference)   # arrays carry no CRS
    print("Saved module6.gdb/steep_share — add it to a Pro map next to the slope raster.")
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

🟢 **`[Python]`**

In [ ]:
# [Python] Anything ArcPy saves as GeoTIFF is just a file to the open-source stack
tif = OUT / "elev_class_arcpy.tif"
if tif.exists():
    with rasterio.open(tif) as src:
        vals, counts = np.unique(src.read(1, masked=True).compressed(), return_counts=True)
        print(f"rasterio opened the ArcPy output: EPSG:{src.crs.to_epsg()}, classes {vals.tolist()}")
else:
    print("No ArcPy output yet — Section 4 builds the same classes with NumPy, so carry on.")

> 🔧 **Try it yourself (ArcGIS Pro):** Change the elevation limit in `suitable_r` from `1000` to `800` and the top `RemapRange` break to match. Re-run and compare the class counts. On JupyterHub, make the same change in Section 4's `BREAKS` list instead.

🟠 **`[Python / ArcPy]`**

In [ ]:
# [Python / ArcPy] Extract by Mask, Project Raster, Resample
if HAS_ARCPY:
    # 1. Clip — Extract by Mask (ArcPy projects the State Plane boundary onto the UTM raster for you)
    ExtractByMask(DEM_A, WRIA_FC).save("dem_wria8")
    ExtractByMask(NLCD_A, WRIA_FC).save("nlcd_wria8")

    # 2. Reproject and 3. resample — clear the snap raster: it's on the UTM grid
    with arcpy.EnvManager(snapRaster=None):
        arcpy.management.ProjectRaster("dem_wria8", "dem_wria8_2926", arcpy.SpatialReference(COURSE_EPSG),
                                       "BILINEAR", "100 100")              # continuous → BILINEAR, 100 ft
        arcpy.management.Resample("nlcd_wria8", "nlcd_wria8_90m", "90 90", "MAJORITY")   # categorical → MAJORITY

    for name in ["dem_wria8", "dem_wria8_2926", "nlcd_wria8_90m"]:
        r_ = arcpy.Raster(name)
        print(f"{name:<16} WKID {r_.spatialReference.factoryCode:<6} "
              f"{r_.meanCellWidth:g} {r_.spatialReference.linearUnitName:<16} {r_.width} × {r_.height}")
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

🟢 **`[Python]`**

In [ ]:
# [Python] Clip both rasters to WRIA 8 with rasterio.mask — the working rasters for tonight
from rasterio.mask import mask as rio_mask

def clip_to(src_path, shapes, dst_path, nodata):
    """Extract by Mask: crop to the shapes' extent and set cells outside them to NoData."""
    with rasterio.open(src_path) as src:
        arr, transform = rio_mask(src, shapes, crop=True, nodata=nodata)
        profile = src.profile | dict(height=arr.shape[1], width=arr.shape[2], transform=transform,
                                     nodata=nodata, compress="deflate")
        try:
            colormap = src.colormap(1)           # keep the NLCD colours for ArcGIS Pro / QGIS
        except ValueError:
            colormap = None
    with rasterio.open(dst_path, "w", **profile) as dst:
        dst.write(arr)
        if colormap:
            dst.write_colormap(1, colormap)
    return dst_path
RAST = Path("module5_rasters")
DEM_TIF, NLCD_TIF = RAST / "wria8_dem_30m.tif", RAST / "wria8_nlcd2021_30m.tif"
DEM_W  = clip_to(DEM_TIF,  wria.geometry, OUT / "dem_wria8.tif",  nodata=-9999.0)   # shapes in the RASTER's CRS
NLCD_W = clip_to(NLCD_TIF, wria.geometry, OUT / "nlcd_wria8.tif", nodata=0)

inspect_raster(DEM_W)        # ← NoData appears: every cell outside the WRIA 8 boundary
inspect_raster(NLCD_W);

In [ ]:
# [Python] Reproject to the State Plane grid, then resample land cover 30 m → 90 m
from rasterio.warp import calculate_default_transform, reproject, Resampling

# ── Reproject: UTM metres → State Plane feet, bilinear because elevation is continuous ──
with rasterio.open(DEM_W) as src:
    transform, width, height = calculate_default_transform(
        src.crs, f"EPSG:{COURSE_EPSG}", src.width, src.height, *src.bounds, resolution=100)  # 100 ft
    profile = src.profile | dict(crs=f"EPSG:{COURSE_EPSG}", transform=transform, width=width, height=height)
    with rasterio.open(OUT / "dem_wria8_2926.tif", "w", **profile) as dst:
        reproject(rasterio.band(src, 1), rasterio.band(dst, 1), resampling=Resampling.bilinear)
inspect_raster(OUT / "dem_wria8_2926.tif")

# ── Resample land cover: the method decides whether the output is still land cover ──
with rasterio.open(NLCD_W) as src:
    lc30, lc_transform, lc_crs = src.read(1), src.transform, src.crs
real_classes = np.unique(lc30[lc30 > 0])                      # the classes that actually exist
print("\n30 m → 90 m land cover:")
for method in [Resampling.mode, Resampling.nearest, Resampling.bilinear]:
    lc90 = np.zeros((lc30.shape[0] // 3, lc30.shape[1] // 3), dtype="uint8")
    reproject(lc30, lc90, src_transform=lc_transform, src_crs=lc_crs, src_nodata=0,
              dst_transform=lc_transform * lc_transform.scale(3), dst_crs=lc_crs, dst_nodata=0,
              resampling=method)
    land = lc90[lc90 > 0]
    print(f"  {method.name:<9} {len(np.unique(land)):>3} distinct values | "
          f"{100 * (~np.isin(land, real_classes)).mean():5.1f}% of cells hold codes that aren't NLCD classes")

🔵 **`[R]`**

In [ ]:
# [R] The same warps with terra: crop() + mask(), project(), aggregate()
wria_v <- vect(wria)                              # sf → SpatVector (already in the rasters' CRS)

dem_w  <- mask(crop(dem,  wria_v), wria_v)        # crop = extent, mask = shape → NoData outside
nlcd_w <- mask(crop(nlcd, wria_v), wria_v)
dem_w

dem_2926 <- project(dem_w, "EPSG:2926", method = "bilinear", res = 100)   # 100 ft, continuous → bilinear
cat("Reprojected DEM: EPSG", crs(dem_2926, describe = TRUE)$code, "|", res(dem_2926)[1], "ft cells\n")

nlcd_90_ok  <- aggregate(nlcd_w, fact = 3, fun = "modal")   # categorical → modal (majority)
nlcd_90_bad <- aggregate(nlcd_w, fact = 3, fun = "mean")    # ❌ averages class codes
cat("modal:", nrow(freq(nlcd_90_ok)), "classes | mean:", nrow(freq(nlcd_90_bad)), "distinct values\n")

plot(dem_w, main = "DEM clipped to WRIA 8 (m)")

> 🔧 **Try it yourself:** In the Python clip cell, swap `wria.geometry` for `wria8.geometry` (the State Plane version) and run it. Unlike silent failures you'll see in Section 6, `rasterio` stops with *"Input shapes do not overlap raster"* — the CRS mismatch again. Swap it back before moving on.

---
## Section 4 — Map Algebra: Local, Focal, Global

Every raster operation tonight fits one of three scopes:

| Scope | Each output cell depends on… | Tonight's examples |
|---|---|---|
| **Local** | the same cell in one or more input rasters | reclassify elevation; `(slope < 15) & (elev < 1000) & (landcover != water)` |
| **Focal** | a moving window around the cell | slope (a 3 × 3 window), 3 × 3 smoothing |
| **Global** | the whole raster (or a whole zone — Sections 5–6) | mean elevation, percent of WRIA 8 that's suitable |

Because the clipped DEM and land cover share one grid (Section 1's checklist), combining them cell by cell is just array arithmetic.

**Slope, three ways, one method.** ArcGIS `Slope`, `terra::terrain()`, and the NumPy cell below all use **Horn's 3 × 3 method**, so their numbers agree.

🟠 **`[Python / ArcPy]`**


In [ ]:
# [Python / ArcPy] Local (Con with two rasters), focal (Focal Statistics), global (raster properties)
if HAS_ARCPY:
    dem_r, nlcd_r = Raster("dem_wria8"), Raster("nlcd_wria8")       # clipped in Section 3
    slope_r = Slope(dem_r, "PERCENT_RISE")

    # LOCAL — Con() is the Raster Calculator's if/else; three rasters combined cell by cell
    suit_r = Con((slope_r < 15) & (dem_r < 1000) & (nlcd_r != 11), 1, 0)
    suit_r.save("suitable_wria8")

    # FOCAL — 3×3 moving-window mean (Focal Statistics tool)
    FocalStatistics(dem_r, NbrRectangle(3, 3, "CELL"), "MEAN", "DATA").save("dem_smooth3")

    # GLOBAL — one number for the whole raster
    arcpy.management.CalculateStatistics("suitable_wria8")
    share = float(arcpy.management.GetRasterProperties("suitable_wria8", "MEAN").getOutput(0))
    print(f"Suitable (slope < 15 %, elevation < 1000 m, not open water): {100 * share:.1f}% of WRIA 8")
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

🟢 **`[Python]`**

In [ ]:
# [Python] LOCAL operations — NumPy arithmetic on aligned arrays
from scipy import ndimage

with rasterio.open(DEM_W) as src:
    dem = src.read(1, masked=True).filled(np.nan)       # NoData → NaN, so it propagates honestly
    profile, cell = src.profile, src.res[0]
with rasterio.open(NLCD_W) as src:
    nlcd = src.read(1)
valid = ~np.isnan(dem)

def horn_slope_pct(z, cell):
    """Slope in % rise from a 3×3 window (Horn's method, as in ArcGIS Slope and terra::terrain)."""
    kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]) / (8 * cell)
    dzdx = ndimage.correlate(z, kx, mode="nearest")
    dzdy = ndimage.correlate(z, kx.T, mode="nearest")
    return np.hypot(dzdx, dzdy) * 100

slope_pct = horn_slope_pct(dem, cell)                    # slope itself is a FOCAL operation

# Reclassify — np.digitize bins every cell (same breaks as the ArcPy RemapRange)
BREAKS = [100, 500, 1000]
elev_class = np.where(valid, np.digitize(dem, BREAKS) + 1, 0).astype("uint8")   # 1–4, 0 = NoData

# Combine three rasters cell by cell → boolean suitability
suitable = (slope_pct < 15) & (dem < 1000) & (nlcd != 11)

print(f"Suitable: {100 * suitable[valid].mean():.1f}% of WRIA 8")
labels = ["lowland  < 100 m", "foothill 100–500 m", "montane  500–1000 m", "subalpine > 1000 m"]
for k, label in enumerate(labels, start=1):
    print(f"  class {k}  {label:<20} {100 * (elev_class == k).sum() / valid.sum():5.1f}% of cells")

def write_like(arr, path, dtype, nodata):
    """Write an array on the DEM grid (same CRS, transform, size)."""
    with rasterio.open(path, "w", **(profile | dict(dtype=dtype, nodata=nodata))) as dst:
        dst.write(arr.astype(dtype), 1)

write_like(np.where(np.isnan(slope_pct), -9999, slope_pct), OUT / "slope_pct.tif", "float32", -9999.0)
write_like(elev_class, OUT / "elev_class.tif", "uint8", 0)
write_like(np.where(valid, suitable, 255), OUT / "suitable.tif", "uint8", 255)

In [ ]:
# [Python] Map the three products side by side
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

fig, ax = plt.subplots(1, 3, figsize=(16, 5.5))
im = ax[0].imshow(np.ma.masked_invalid(dem), cmap="terrain")
plt.colorbar(im, ax=ax[0], shrink=0.75, label="Elevation (m)")
ax[0].set_title("DEM clipped to WRIA 8")

class_colors = ["#1b7837", "#a6dba0", "#c2a5cf", "#762a83"]
ax[1].imshow(np.ma.masked_equal(elev_class, 0), cmap=ListedColormap(class_colors), vmin=1, vmax=4)
ax[1].legend(handles=[Patch(color=c, label=l) for c, l in zip(class_colors, labels)],
             loc="lower left", fontsize=8)
ax[1].set_title("Reclassified elevation (local)")

ax[2].imshow(np.ma.masked_where(~valid, suitable), cmap=ListedColormap(["#d9d9d9", "#4b2e83"]))
ax[2].legend(handles=[Patch(color="#4b2e83", label="suitable"), Patch(color="#d9d9d9", label="not suitable")],
             loc="lower left", fontsize=8)
ax[2].set_title("Slope < 15 %, elev < 1000 m, not water")
for a in ax:
    a.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# [Python] FOCAL and GLOBAL operations — and the NoData gotcha
def focal_mean(z, size=3, na_rm=False):
    """Moving-window mean. na_rm=False: NoData if the window touches any NoData (like terra::focal);
    na_rm=True: average whatever valid cells the window holds (like ArcGIS Focal Statistics 'DATA')."""
    ok_ = ~np.isnan(z)
    total = ndimage.uniform_filter(np.where(ok_, z, 0.0), size=size, mode="nearest")
    share = ndimage.uniform_filter(ok_.astype("float64"), size=size, mode="nearest")   # fraction of valid cells
    mean  = total / np.where(share > 0, share, 1)
    keep  = share > 0 if na_rm else share > 1 - 1e-9              # some valid cells vs. all valid cells
    return np.where(keep, mean, np.nan)

smooth = focal_mean(dem, size=3)
print(f"NoData cells before smoothing:  {np.isnan(dem).sum():,}")
print(f"After 3×3 mean (na_rm=False):   {np.isnan(smooth).sum():,}   ← NoData grows by a one-cell ring")
print(f"After 3×3 mean (na_rm=True):    {np.isnan(focal_mean(dem, 3, na_rm=True)).sum():,}   ← edge cells kept")
raw = ndimage.uniform_filter(dem, size=3)                         # ❌ scipy filter on the unmasked array
print(f"scipy uniform_filter on raw NaN: {np.isnan(raw).mean():.0%} of cells NaN — it uses running sums, "
      "so one NaN wipes out whole rows. Handle NoData first.")

# GLOBAL — one number from the whole raster (nan-aware functions skip NoData)
print(f"\nMean elevation {np.nanmean(dem):.0f} m | highest {np.nanmax(dem):.0f} m | "
      f"90th-percentile slope {np.nanpercentile(slope_pct, 90):.0f}%")
print(f"np.mean without nan-awareness: {np.mean(dem)}   ← one NaN poisons a global statistic")

🔵 **`[R]`**

In [ ]:
# [R] Local, focal and global map algebra with terra
slope_deg <- terrain(dem_w, v = "slope", unit = "degrees")      # Horn 3×3, as in ArcGIS Slope
slope_pct <- tan(slope_deg * pi / 180) * 100                     # degrees → % rise

# LOCAL — classify() with a from / to / becomes matrix; right = FALSE → [from, to)
rcl <- matrix(c(-Inf,  100, 1,
                 100,  500, 2,
                 500, 1000, 3,
                1000,  Inf, 4), ncol = 3, byrow = TRUE)
elev_class <- classify(dem_w, rcl, right = FALSE)
suitable   <- (slope_pct < 15) & (dem_w < 1000) & (nlcd_w != 11)   # three rasters, cell by cell
cat(sprintf("Suitable: %.1f%% of WRIA 8\n", 100 * global(suitable, "mean", na.rm = TRUE)[[1]]))
print(freq(elev_class))

# FOCAL — focal() keeps NA by default, so NoData spreads one cell inward
dem_smooth <- focal(dem_w, w = 3, fun = "mean")
cat("NA cells before:", global(is.na(dem_w), "sum")[[1]],
    "| after 3×3 focal:", global(is.na(dem_smooth), "sum")[[1]], "\n")

# GLOBAL
print(global(dem_w, c("mean", "max"), na.rm = TRUE))

par(mfrow = c(1, 2))
plot(elev_class, col = c("#1b7837", "#a6dba0", "#c2a5cf", "#762a83"), main = "Elevation classes")
plot(suitable, col = c("#d9d9d9", "#4b2e83"), main = "Suitable")
par(mfrow = c(1, 1))

> 🔧 **Try it yourself:** In the Python local-operations cell, change the slope limit from `15` to `10` and re-run that cell and the map cell. How much does the suitable share drop? Then change `size=3` to `size=9` in the `focal_mean()` call — how many more NoData cells does the bigger window create, and why?

---
## Section 5 — Zonal Statistics: the ESRI Path

Zonal statistics collapse a raster into **one row per polygon** — mean elevation *by watershed*, percent forest *by watershed*. It's where the vector work of Modules 3–4 and the raster work of Modules 5–6 meet: the **zones** are polygons, the **values** are a raster.

| ESRI tool | What it answers | Raster type |
|---|---|---|
| **Zonal Statistics as Table** | mean / min / max / std of values in each zone | continuous (DEM, slope) |
| **Tabulate Area** | area of each class in each zone | categorical (land cover) |
| **Join Field** | attach the table back onto the polygons | — |

Both ESRI paths below are 🏠 **LOCAL ONLY**: ArcPy needs Spatial Analyst; the R-ArcGIS Bridge needs ArcGIS Pro and `arcgisbinding`.

🟠 **`[Python / ArcPy]`**

In [ ]:
# [Python / ArcPy] Zonal Statistics as Table + Tabulate Area + Join Field, then straight into pandas
if HAS_ARCPY:
    import pandas as pd
    ws_fc = arcpy.management.CopyFeatures(WS_FC, "watersheds_zonal")[0]   # never write into the course gdb

    # Continuous: elevation statistics per watershed (zones in State Plane, raster in UTM — ArcPy handles it)
    ZonalStatisticsAsTable(ws_fc, "WATERSHED_ID", "dem_wria8", "zonal_dem", "DATA", "MEAN_STD")
    arcpy.management.JoinField(ws_fc, "WATERSHED_ID", "zonal_dem", "WATERSHED_ID", ["MEAN", "STD"])

    # Categorical: area of every NLCD class in every watershed (one column per class: VALUE_11, VALUE_41, …)
    TabulateArea(ws_fc, "WATERSHED_ID", "nlcd_wria8", "Value", "lc_area", 30)

    # ArcPy tables → pandas DataFrames in one line each
    elev_tbl = pd.DataFrame(arcpy.da.TableToNumPyArray(ws_fc, ["WATERSHED_ID", "WATERSHED_NAME", "MEAN", "STD"]))
    lc_tbl   = pd.DataFrame(arcpy.da.TableToNumPyArray("lc_area", "*"))
    forest   = [c for c in lc_tbl.columns if c in ("VALUE_41", "VALUE_42", "VALUE_43")]
    lc_cols  = [c for c in lc_tbl.columns if c.startswith("VALUE_")]
    lc_tbl["pct_forest"] = (100 * lc_tbl[forest].sum(axis=1) / lc_tbl[lc_cols].sum(axis=1)).round(1)
    arc_zonal = elev_tbl.merge(lc_tbl[["WATERSHED_ID", "pct_forest"]], on="WATERSHED_ID")   # join on the key, not row order
    print(arc_zonal.sort_values("MEAN", ascending=False).round(1).to_string(index=False))
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

🔵 **`[R]`**

In [ ]:
# [R] 🏠 LOCAL ONLY — R-ArcGIS Bridge: read Pro data into R, compute, write back to a geodatabase
# The Bridge MOVES data; the computing happens in R (here with exactextractr).
if (requireNamespace("arcgisbinding", quietly = TRUE)) {
  library(arcgisbinding)
  arc.check_product()  
  # install.packages("exactextractr")   # if you don't have it yet                                          # license handshake with ArcGIS Pro

  ws_arc <- arc.data2sf(arc.select(arc.open(file.path(normalizePath(WS_GDB), "watersheds"))))
  ws_arc <- st_transform(ws_arc, RASTER_EPSG)                    # R tools need matching CRSs
  ws_arc$ELEV_MEAN <- exactextractr::exact_extract(dem_w, ws_arc, "mean", progress = FALSE)

  out_gdb <- file.path(normalizePath(OUT), "module6.gdb")        # created by the ArcPy setup cell
  if (dir.exists(out_gdb)) {
    arc.write(file.path(out_gdb, "ws_elev_bridge"), ws_arc, overwrite = TRUE)
    cat("Wrote ws_elev_bridge to module6.gdb — open it in ArcGIS Pro\n")
  }
  print(st_drop_geometry(ws_arc)[, c("WATERSHED_NAME", "ELEV_MEAN")])
} else {
  cat("arcgisbinding not installed here — reference cell. Run it in RStudio/Positron with ArcGIS Pro.\n")
}

---
## Section 6 — Zonal Statistics: the Open-Source Path

Same table, no license: `rasterstats` (Python) and `exactextractr` (R) run anywhere, including JupyterHub. Two things to notice:

- `categorical=True` (Python) and `"frac"` (R) return **class counts / fractions** — the Tabulate Area equivalent. Never take the *mean* of a land cover code.
- Unlike ArcPy, these tools **do not reproject on the fly**. Zones and raster must already share a CRS — and if they don't, you get no error, just empty or wrong results (the cell after the next one shows it).

🟢 **`[Python]`**

In [ ]:
# [Python] rasterstats — zonal statistics on the clipped rasters
from rasterstats import zonal_stats
import pandas as pd

FOREST    = {41, 42, 43}           # deciduous, evergreen, mixed forest
DEVELOPED = {21, 22, 23, 24}       # developed, open space → high intensity

elev  = pd.DataFrame(zonal_stats(ws, str(DEM_W), stats=["mean", "min", "max", "std"]))
slope = pd.DataFrame(zonal_stats(ws, str(OUT / "slope_pct.tif"), stats=["mean"]))
lc    = zonal_stats(ws, str(NLCD_W), categorical=True)          # one {class: cell count} dict per watershed

def pct(counts, classes):
    return round(100 * sum(n for k, n in counts.items() if k in classes) / sum(counts.values()), 1)

zones = ws[["watershed_id", "watershed_name", "geometry"]].copy()
zones["elev_mean"]     = elev["mean"].round(0)
zones["elev_range"]    = (elev["max"] - elev["min"]).round(0)
zones["slope_mean"]    = slope["mean"].round(1)
zones["pct_forest"]    = [pct(d, FOREST) for d in lc]
zones["pct_developed"] = [pct(d, DEVELOPED) for d in lc]

print(zones.drop(columns="geometry").sort_values("elev_mean", ascending=False).to_string(index=False))

In [ ]:
# [Python] ⚠️ The silent failure: the same call with the zones still in State Plane feet
bad = pd.DataFrame(zonal_stats(watersheds, str(DEM_W), stats=["mean", "count"]))   # watersheds = EPSG:2926
print(bad.head())
print("\nNo error, no warning — every watershed 'contains' zero cells. Always check CRSs before zonal stats.")

🔵 **`[R]`**

In [ ]:
# [R] exactextractr — area-weighted zonal statistics (partial edge cells counted by their coverage)
library(exactextractr)

zonal <- exact_extract(dem_w, ws, c("mean", "min", "max", "stdev"), progress = FALSE)
zones_r <- ws[, c("watershed_id", "watershed_name")]
zones_r$elev_mean  <- round(zonal$mean)
zones_r$elev_range <- round(zonal$max - zonal$min)
zones_r$slope_mean <- round(exact_extract(slope_pct, ws, "mean", progress = FALSE), 1)

# Categorical: "frac" returns one column per class (frac_11, frac_21, …) — the Tabulate Area equivalent
lc_frac <- exact_extract(nlcd_w, ws, "frac", progress = FALSE)
forest_cols <- intersect(c("frac_41", "frac_42", "frac_43"), names(lc_frac))
zones_r$pct_forest <- round(100 * rowSums(lc_frac[, forest_cols]), 1)

print(st_drop_geometry(zones_r)[order(-zones_r$elev_mean), ], row.names = FALSE)

# terra-only alternative (no extra package):  terra::zonal(dem_w, vect(ws), fun = "mean", na.rm = TRUE)

> 🔧 **Try it yourself:** Add `"median"` and `"count"` to the elevation `zonal_stats()` call. In which watershed do the mean and median differ most — and what does that say about its terrain? Then add `52` (shrub/scrub) to `FOREST` and see which watershed's percentage moves most.

---
## Section 7 — Rasterize and Vectorize: Crossing the Divide

- **Rasterize (vector → raster):** burn polygon IDs onto the raster grid. Zonal logic then runs at array speed — one `bincount` instead of 15 polygon masks — and the zone grid can be used in map algebra.
- **Vectorize (raster → vector):** trace class boundaries into polygons, so results can join attribute tables, feed web maps, or land in a geodatabase.

**Rule:** vectorize **categorical** rasters only. Polygonizing a continuous DEM gives one polygon per cell — reclassify first (Section 4), then vectorize.

ESRI tools: **Polygon to Raster**, **Raster to Polygon**.

🟠 **`[Python / ArcPy]`**

In [ ]:
# [Python / ArcPy] Polygon to Raster and Raster to Polygon
if HAS_ARCPY:
    # Polygon to Raster writes in the FEATURES' CRS unless you say otherwise — force the raster grid
    with arcpy.EnvManager(outputCoordinateSystem=arcpy.SpatialReference(RASTER_EPSG), snapRaster=DEM_A):
        arcpy.conversion.PolygonToRaster(WS_FC, "WATERSHED_ID", "ws_grid", "CELL_CENTER", "NONE", 30)

    # Vectorize the elevation classes inside one watershed
    arcpy.management.MakeFeatureLayer(WS_FC, "issaquah", "WATERSHED_NAME = 'Issaquah Creek'")
    ExtractByMask("elev_class_arcpy", "issaquah").save("elev_class_issaquah")
    arcpy.conversion.RasterToPolygon("elev_class_issaquah", "issaquah_elev_polys", "SIMPLIFY", "Value")
    print("Issaquah Creek elevation-class polygons:", arcpy.management.GetCount("issaquah_elev_polys")[0])
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

🟢 **`[Python]`**

In [ ]:
# [Python] rasterio.features — rasterize the watersheds, vectorize the elevation classes
from rasterio import features
from shapely.geometry import shape

# ── Vector → raster: burn an integer ID (1–15; 0 = outside) onto the DEM grid ──
ws_num  = np.arange(1, len(ws) + 1)
ws_grid = features.rasterize(zip(ws.geometry, ws_num), out_shape=dem.shape,
                             transform=profile["transform"], fill=0, dtype="uint8")

# Zonal mean at grid speed: one pass with np.bincount
ok     = (ws_grid > 0) & valid
sums   = np.bincount(ws_grid[ok], weights=dem[ok], minlength=len(ws) + 1)
counts = np.bincount(ws_grid[ok], minlength=len(ws) + 1)
check = pd.DataFrame({"watershed": ws["watershed_name"].values,
                      "mean_via_rasterize": (sums / np.maximum(counts, 1))[1:].round(0),
                      "mean_via_rasterstats": zones["elev_mean"].values})
print(check.to_string(index=False))

# ── Raster → vector: elevation classes inside Issaquah Creek → dissolved polygons ──
issaquah = ws[ws["watershed_name"] == "Issaquah Creek"]
with rasterio.open(OUT / "elev_class.tif") as src:
    cls, cls_transform = rio_mask(src, issaquah.geometry, crop=True, nodata=0)

polys = [{"geometry": shape(geom), "elev_class": int(value)}
         for geom, value in features.shapes(cls[0], mask=cls[0] > 0, transform=cls_transform)]
class_gdf = gpd.GeoDataFrame(polys, crs=RASTER_EPSG).dissolve(by="elev_class", as_index=False)
class_gdf["area_km2"] = (class_gdf.area / 1e6).round(1)
class_gdf.to_file(OUT / "issaquah_elev_classes.gpkg", layer="elev_classes", driver="GPKG")
print("\nIssaquah Creek by elevation class:")
print(class_gdf[["elev_class", "area_km2"]].to_string(index=False))

🔵 **`[R]`**

In [ ]:
# [R] terra: rasterize() and as.polygons()
ws_v    <- vect(ws)
ws_grid <- rasterize(ws_v, dem_w, field = "watershed_id")      # zones burned onto the DEM grid
print(zonal(dem_w, ws_grid, fun = "mean", na.rm = TRUE))       # zonal stats from two rasters

iss       <- ws_v[ws_v$watershed_name == "Issaquah Creek", ]
cls_iss   <- mask(crop(elev_class, iss), iss)
class_sf  <- st_as_sf(as.polygons(cls_iss))                    # dissolves by value automatically
class_sf$area_km2 <- round(as.numeric(st_area(class_sf)) / 1e6, 1)
print(st_drop_geometry(class_sf))
st_write(class_sf, file.path(OUT, "issaquah_elev_classes_r.gpkg"), delete_dsn = TRUE, quiet = TRUE)

> 🔧 **Try it yourself:** Change `"Issaquah Creek"` to `"Rex River"` (or any name from the zonal table) in the Python vectorize block and re-run. Then vectorize `suitable.tif` instead of `elev_class.tif` — what share of that watershed is suitable, in km²?

---
## Section 8 — Multidimensional Arrays: xarray and stars

NetCDF and HDF files hold rasters with **extra labelled dimensions** — usually time. With `xarray` (Python) and `stars` (R) you write `sel(time="2024-07")` instead of keeping track of which of 24 files is July 2024.

**Tonight's cube:** 24 monthly NDVI grids (2023–2024) on a 300 m version of the WRIA 8 grid. The values are **simulated from the real NLCD classes**: forests green up and stay high, developed land stays low, water stays negative, and one block of forest is harvested in spring 2024. The cell below builds it once. Module 10 swaps in real satellite time series.

ESRI equivalents: multidimensional rasters in ArcGIS Pro — **Make Multidimensional Raster Layer**, **Aggregate Multidimensional Raster**.

🟢 **`[Python]`**

In [ ]:
# [Python] Build tonight's NDVI cube (simulated values on the real WRIA 8 grid) → NetCDF
import xarray as xr

from pyproj import CRS
from rasterio.windows import Window

with rasterio.open(NLCD_W) as src:
    f = 10                                                           # 30 m → exactly 300 m cells
    h, w = (src.height // f) * f, (src.width // f) * f               # whole 300 m blocks only
    lc300 = src.read(1, window=Window(0, 0, w, h), out_shape=(h // f, w // f), resampling=Resampling.mode)
    t300  = src.transform * src.transform.scale(f)
    cf_crs = CRS.from_user_input(src.crs).to_cf()                    # CF grid-mapping attributes (GDAL, stars, ArcGIS)

rng   = np.random.default_rng(6)
peak  = np.select([np.isin(lc300, [41, 42, 43, 90]), np.isin(lc300, [52, 71, 81, 82, 95]),
                   np.isin(lc300, [21, 22]), np.isin(lc300, [23, 24, 31])],
                  [0.85, 0.70, 0.50, 0.25], default=-0.05)           # typical summer NDVI by cover type
times  = pd.date_range("2023-01-01", periods=24, freq="MS") + pd.Timedelta(days=14)
season = 0.5 + 0.5 * np.cos(2 * np.pi * (times.month.values - 7) / 12)   # 1 in July, 0 in January
cube   = peak[None] * (0.55 + 0.45 * season[:, None, None]) + rng.normal(0, 0.02, (24, *lc300.shape))

# A forest harvest: part of Taylor Creek-Cedar River loses most of its canopy from April 2024
taylor = features.rasterize(ws.loc[ws["watershed_name"] == "Taylor Creek-Cedar River", "geometry"],
                            out_shape=lc300.shape, transform=t300, fill=0, dtype="uint8").astype(bool)
cols = np.arange(lc300.shape[1])[None, :]
harvest = taylor & (cols < np.percentile(np.where(taylor)[1], 45)) & np.isin(lc300, [41, 42, 43])
after = np.asarray(times >= "2024-04-01")
cube[after] = np.where(harvest, cube[after] * 0.45, cube[after])
cube[:, lc300 == 0] = np.nan                                        # outside WRIA 8

x = t300.c + (np.arange(lc300.shape[1]) + 0.5) * t300.a
y = t300.f + (np.arange(lc300.shape[0]) + 0.5) * t300.e
ds = xr.Dataset({"ndvi": (("time", "y", "x"), cube.astype("float32"),
                          {"long_name": "NDVI (simulated)", "grid_mapping": "spatial_ref"})},
                coords={"time": times, "y": ("y", y, {"units": "m", "standard_name": "projection_y_coordinate"}),
                        "x": ("x", x, {"units": "m", "standard_name": "projection_x_coordinate"}),
                        "spatial_ref": ((), 0, cf_crs | {"spatial_ref": cf_crs["crs_wkt"]})})
NDVI_NC = OUT / "ndvi_monthly_wria8.nc"
ds.to_netcdf(NDVI_NC)
print(f"Wrote {NDVI_NC}: {dict(ds.sizes)}  ({harvest.sum()} harvested 300 m cells)")

In [ ]:
# [Python] xarray — open the cube, slice by label, reduce across dimensions
ds   = xr.open_dataset(NDVI_NC)
print(ds)                                                  # dims, coords, variables — always look first
ndvi = ds["ndvi"]

july_2024 = ndvi.sel(time="2024-07").squeeze("time")       # label-based: "July 2024", not index 18
growing   = ndvi.sel(time=slice("2024-04", "2024-09")).mean(dim="time")   # growing-season mean per pixel
curve     = ndvi.mean(dim=["y", "x"])                      # one number per month: the WRIA 8 curve
by_month  = ndvi.groupby("time.month").mean(dim="time")    # 12 'average month' grids from 2 years

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
july_2024.plot(ax=ax[0], cmap="YlGn", vmin=-0.1, vmax=0.9)
ax[0].set_title("NDVI — July 2024")
growing.plot(ax=ax[1], cmap="YlGn", vmin=-0.1, vmax=0.9)
ax[1].set_title("Growing-season mean, Apr–Sep 2024")
curve.plot(ax=ax[2], marker="o", color="#4b2e83")
ax[2].set_title("WRIA 8 mean NDVI by month")
for a in ax[:2]:
    a.set_axis_off()
plt.tight_layout()
plt.show()
print("Greenest month on average:", int(by_month.mean(dim=["y", "x"]).idxmax()))

🟠 **`[Python / ArcPy]`**

In [ ]:
# [Python / ArcPy] The same NetCDF as an ArcGIS multidimensional raster
if HAS_ARCPY:
    try:
        NDVI_NC = OUT / "ndvi_monthly_wria8.nc"
        nc = str(NDVI_NC.resolve())
        md_r = arcpy.Raster(nc, True)                          # True = open as multidimensional
        dims = md_r.getDimensionNames("ndvi")                  # dimensions belong to a variable
        print("Variables:", md_r.variableNames, "| dimensions of ndvi:", dims)
        time_dim = next(d for d in dims if "time" in d.lower())   # usually "StdTime" in ArcGIS

        # Two yearly means from 24 monthly slices
        yearly = AggregateMultidimensionalRaster(nc, time_dim, "MEAN", "ndvi", "INTERVAL_KEYWORD", "YEARLY")
        yearly.save(str((OUT / "ndvi_yearly.crf").resolve()))   # multidimensional output → CRF
        print("Saved ndvi_yearly.crf — add it in Pro and use the time slider.")
    except Exception as e:
        print("ArcGIS couldn't read the cube:", e)
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

🔵 **`[R]`**

In [ ]:
# [R] stars — the same cube with dimension-aware R (run the Python cube-builder cell first)
library(stars)
install.packages("ncmeta")   # if you don't have it yet
nc <- read_ncdf(file.path(OUT, "ndvi_monthly_wria8.nc"), var = "ndvi", proxy = FALSE)
st_crs(nc) <- st_crs(RASTER_EPSG)                      # NetCDF stores the CRS as CF parameters — name the EPSG code
nc                                                     # note the x / y / time dimensions
times <- st_get_dimension_values(nc, "time")

july_2024 <- dplyr::slice(nc, time, which(format(times, "%Y-%m") == "2024-07"))
growing   <- st_apply(nc[, , , which(times >= as.POSIXct("2024-04-01") & times < as.POSIXct("2024-10-01"))],
                      c("x", "y"), mean, na.rm = TRUE)     # reduce over time, keep x and y
curve     <- st_apply(nc, "time", mean, na.rm = TRUE)      # reduce over x and y, keep time

plot(july_2024, main = "NDVI — July 2024", col = hcl.colors(20, "YlGn", rev = TRUE))
plot(as.Date(times), curve[[1]], type = "b", xlab = "", ylab = "mean NDVI", main = "WRIA 8 mean NDVI by month")

> 🔧 **Try it yourself:** Change `"2024-07"` to `"2023-07"` and compare the two July maps — can you find the 2024 harvest block? Then change the growing season to `slice("2024-06", "2024-08")` (summer only).

---
## Section 9 — Guided Lab: Elevation & Suitability by Watershed

Back to the original workflow presented at the beginning of the lab. Every step below reuses a pattern from Sections 1–6, so the job is **run it, read the output, then make the 🔧 change** and run it again.

| Step | Operation | Pattern from |
|---|---|---|
| 1 | Load & inspect the clipped rasters and the watersheds | Section 1 |
| 2 | Reclassify the DEM into elevation classes | Section 4 (local) |
| 3 | Map algebra: slope + elevation + land cover → suitability | Section 4 (focal + local) |
| 4 | Zonal statistics per watershed | Section 6 |
| 5 | Export a GeoPackage + summary CSV for ArcGIS Pro | Module 3 |

Python steps come one cell at a time; the R version is a single cell after Step 5, and the ArcPy version follows it.

🟢 **`[Python]`**

In [ ]:
# [Python] Lab Step 1 — Load & inspect (the Module 5 habit, unchanged)
LAB_DEM, LAB_LC = OUT / "dem_wria8.tif", OUT / "nlcd_wria8.tif"      # clipped to WRIA 8 in Section 3
inspect_raster(LAB_DEM)
inspect_raster(LAB_LC)

lab_ws = gpd.read_file(WS_GDB, layer="watersheds").rename(columns=str.lower).to_crs(RASTER_EPSG)
print(f"{len(lab_ws)} watersheds in EPSG:{lab_ws.crs.to_epsg()} — same CRS as the rasters ✔")

> 🔧 **Change it:** Keep only the Cedar River watersheds by adding `lab_ws = lab_ws[lab_ws["watershed_name"].str.contains("Cedar")]` at the end of the cell. Steps 2–5 then report on just those (run the cell again without the filter to go back to all 15).

In [ ]:
# [Python] Lab Step 2 — Reclassify the DEM into elevation classes
LAB_BREAKS = [100, 500, 1000]                       # metres: lowland | foothill | montane | subalpine

with rasterio.open(LAB_DEM) as src:
    lab_dem  = src.read(1, masked=True).filled(np.nan)
    lab_prof = src.profile
    lab_cell = src.res[0]
lab_valid = ~np.isnan(lab_dem)
lab_class = np.where(lab_valid, np.digitize(lab_dem, LAB_BREAKS) + 1, 0).astype("uint8")

share = pd.Series(lab_class[lab_valid]).value_counts(normalize=True).sort_index() * 100
print("Share of WRIA 8 in each elevation class (%):")
print(share.round(1).to_string())

> 🔧 **Change it:** Set `LAB_BREAKS = [200, 600, 1200]` and re-run. Add a fourth break (e.g. `1500`) — how many classes do you get now, and why one more than the number of breaks?

In [ ]:
# [Python] Lab Step 3 — Map algebra: three rasters → one suitability raster
SLOPE_MAX = 15          # % rise
ELEV_MAX  = 1000        # metres
EXCLUDE   = [11]        # NLCD classes to leave out (11 = open water)

with rasterio.open(LAB_LC) as src:
    lab_lc = src.read(1)

lab_slope = horn_slope_pct(lab_dem, lab_cell)                         # focal (Section 4)
lab_suit  = (lab_slope < SLOPE_MAX) & (lab_dem < ELEV_MAX) & ~np.isin(lab_lc, EXCLUDE)   # local
print(f"Suitable: {100 * lab_suit[lab_valid].mean():.1f}% of WRIA 8")

for arr, name, dtype, nodata in [
        (np.where(np.isnan(lab_slope), -9999, lab_slope), "lab_slope.tif",    "float32", -9999.0),
        (np.where(lab_valid, lab_suit, 255),               "lab_suitable.tif", "uint8",   255)]:
    with rasterio.open(OUT / name, "w", **(lab_prof | dict(dtype=dtype, nodata=nodata))) as dst:
        dst.write(arr.astype(dtype), 1)

> 🔧 **Change it:** Leave developed land out too: `EXCLUDE = [11, 21, 22, 23, 24]`. Then try `SLOPE_MAX = 10`. Re-run Step 3 *and* Step 4 after each change and watch the ranking move.

In [ ]:
# [Python] Lab Step 4 — Zonal statistics per watershed
z_elev  = pd.DataFrame(zonal_stats(lab_ws, str(LAB_DEM),                stats=["mean"]))
z_slope = pd.DataFrame(zonal_stats(lab_ws, str(OUT / "lab_slope.tif"),    stats=["mean"]))
z_suit  = pd.DataFrame(zonal_stats(lab_ws, str(OUT / "lab_suitable.tif"), stats=["mean"]))  # mean of 0/1 = share
z_lc    = zonal_stats(lab_ws, str(LAB_LC), categorical=True)

lab_ws["elev_mean"]    = z_elev["mean"].round(0).values
lab_ws["slope_mean"]   = z_slope["mean"].round(1).values
lab_ws["pct_forest"]   = [pct(d, FOREST) for d in z_lc]
lab_ws["pct_suitable"] = (100 * z_suit["mean"]).round(1).values

summary = (lab_ws.drop(columns="geometry")
           [["watershed_id", "watershed_name", "elev_mean", "slope_mean", "pct_forest", "pct_suitable"]]
           .sort_values("pct_suitable", ascending=False))
print(summary.to_string(index=False))

ax = lab_ws.plot(column="pct_suitable", cmap="Purples", edgecolor="grey", legend=True, figsize=(7, 7),
                 legend_kwds={"label": "% suitable (slope, elevation, not water)", "shrink": 0.6})
ax.set_title("Suitable share by watershed"); ax.set_axis_off()
plt.show()

> 🔧 **Change it:** Add `"median"` to the slope `stats=[…]` list and a `lab_ws["slope_median"]` column. Then sort `summary` by `pct_forest` instead of `pct_suitable`.

In [ ]:
# [Python] Lab Step 5 — Export for ArcGIS Pro (and anyone else)
lab_ws.to_file(OUT / "module6_lab.gpkg", layer="watershed_summary", driver="GPKG")
summary.to_csv(OUT / "module6_lab_summary.csv", index=False)

print("Wrote:")
for f in ["module6_lab.gpkg", "module6_lab_summary.csv", "lab_suitable.tif", "lab_slope.tif"]:
    print(f"  {OUT / f}")
print("In ArcGIS Pro: add the GeoPackage layer, symbolize pct_suitable, and drop lab_suitable.tif on top.")

> 🔧 **Change it:** Change `layer="watershed_summary"` to `layer="cedar_summary"` after running with the Cedar-only filter from Step 1 — one GeoPackage can hold both versions side by side.

In [ ]:
# [R] Lab Steps 1–5 — the same request with terra + exactextractr
# Step 1 — load & inspect (dem_w and nlcd_w were clipped to WRIA 8 in Section 3)
lab_dem <- dem_w
lab_lc  <- nlcd_w
lab_dem
lab_ws  <- st_transform(watersheds, RASTER_EPSG)

# Step 2 — reclassify
LAB_BREAKS <- c(100, 500, 1000)
lab_class  <- classify(lab_dem, cbind(c(-Inf, LAB_BREAKS), c(LAB_BREAKS, Inf), 1:4), right = FALSE)
print(round(100 * prop.table(table(values(lab_class, na.rm = TRUE))), 1))

# Step 3 — map algebra
SLOPE_MAX <- 15; ELEV_MAX <- 1000; EXCLUDE <- c(11)
lab_slope <- tan(terrain(lab_dem, v = "slope", unit = "radians")) * 100
lab_suit  <- (lab_slope < SLOPE_MAX) & (lab_dem < ELEV_MAX) & !(lab_lc %in% EXCLUDE)

# Step 4 — zonal statistics
lab_ws$elev_mean    <- round(exact_extract(lab_dem,   lab_ws, "mean", progress = FALSE))
lab_ws$slope_mean   <- round(exact_extract(lab_slope, lab_ws, "mean", progress = FALSE), 1)
lab_ws$pct_suitable <- round(100 * exact_extract(lab_suit, lab_ws, "mean", progress = FALSE), 1)
fr <- exact_extract(lab_lc, lab_ws, "frac", progress = FALSE)
lab_ws$pct_forest   <- round(100 * rowSums(fr[, intersect(c("frac_41", "frac_42", "frac_43"), names(fr))]), 1)

summary_r <- st_drop_geometry(lab_ws)[order(-lab_ws$pct_suitable),
                                       c("watershed_name", "elev_mean", "slope_mean", "pct_forest", "pct_suitable")]
print(summary_r, row.names = FALSE)

# Step 5 — export
st_write(lab_ws, file.path(OUT, "module6_lab_r.gpkg"), layer = "watershed_summary", delete_dsn = TRUE, quiet = TRUE)
write.csv(summary_r, file.path(OUT, "module6_lab_summary_r.csv"), row.names = FALSE)
cat("Wrote module6_lab_r.gpkg and module6_lab_summary_r.csv to", OUT, "\n")
# R and Python differ by a few tenths: exact_extract weights partial edge cells; rasterstats counts cell centres.

🟠 **`[Python / ArcPy]`**

In [ ]:
# [Python / ArcPy] The whole lab in ArcPy — Steps 1–5 in a dozen lines
if HAS_ARCPY:
    dem_r   = ExtractByMask(DEM_A,  WRIA_FC)                                        # 1  clip
    nlcd_r  = ExtractByMask(NLCD_A, WRIA_FC)
    class_r = Reclassify(dem_r, "Value", RemapRange([[-10, 100, 1], [100, 500, 2],   # 2  reclassify
                                                     [500, 1000, 3], [1000, 2000, 4]]))
    slope_r = Slope(dem_r, "PERCENT_RISE")                                          # 3  map algebra
    suit_r  = Con((slope_r < 15) & (dem_r < 1000) & (nlcd_r != 11), 1, 0)

    lab_fc = arcpy.management.CopyFeatures(WS_FC, "lab_watersheds")[0]              # 4  zonal stats
    for value_r, name in [(dem_r, "ELEV"), (slope_r, "SLOPE"), (suit_r, "SUIT")]:
        ZonalStatisticsAsTable(lab_fc, "WATERSHED_ID", value_r, f"zs_{name}", "DATA", "MEAN")
        arcpy.management.AlterField(f"zs_{name}", "MEAN", f"{name}_MEAN")
        arcpy.management.JoinField(lab_fc, "WATERSHED_ID", f"zs_{name}", "WATERSHED_ID", [f"{name}_MEAN"])

    arc_summary = pd.DataFrame(arcpy.da.TableToNumPyArray(
        lab_fc, ["WATERSHED_NAME", "ELEV_MEAN", "SLOPE_MEAN", "SUIT_MEAN"]))
    arc_summary["PCT_SUITABLE"] = (100 * arc_summary.pop("SUIT_MEAN")).round(1)
    arc_summary = arc_summary.sort_values("PCT_SUITABLE", ascending=False)
    arc_summary.to_csv(OUT / "module6_lab_summary_arcpy.csv", index=False)        # 5  export
    print(arc_summary.round(1).to_string(index=False))
    print("lab_watersheds (with the joined columns) is in module6.gdb — ready to symbolize in Pro.")
else:
    print("ArcPy reference cell — skipped (no arcpy in this environment).")

---
## Debrief — Common Raster Gotchas

| ✗ Gotcha | What you see | ✓ Fix |
|---|---|---|
| **NoData propagates** through map algebra | A focal window or a plain `np.mean` turns valid cells into NaN | Use NaN-aware functions (`np.nanmean`, `na.rm = TRUE`); check NoData counts before and after (Section 4) |
| **CRS mismatch** in zonal statistics | Empty or plausible-looking wrong numbers — no error | Reproject the vectors to the raster's CRS first (Section 6) — ArcPy hides this by projecting on the fly |
| **Wrong resampling** for the data type | Land cover full of codes that aren't classes | `nearest` / `mode` / `MAJORITY` for categorical; `bilinear` / `cubic` for continuous (Section 3) |
| **Eager-loading** a big NetCDF | Memory errors on multi-year stacks | Open lazily with chunks — `xr.open_dataset(path, chunks={})` uses Dask (Module 10) |

### Reflection
Jot a sentence or two for each, in a markdown cell below:

1. Which of tonight's operations replaces something you do by hand in ArcGIS Pro — and roughly how many hours a month would scripting it save?
2. Where did the local / focal / global framing help you pick a tool?
3. What surprised you about the zonal-statistics CRS failure?
4. What question do you still have about raster operations?

*Your reflection notes:*

---
## 🍍 Self-Check Questions

1. Is slope a local, focal, or global operation? What about "percent of WRIA 8 that's suitable"?
2. Write the one-line `arcpy.sa` expression for "slope under 15 % **and** elevation under 1000 m".
3. Name the two functions that move a raster between ArcPy and NumPy. What does `NumPyArrayToRaster()` need that a plain array doesn't carry?
4. You resample NLCD from 30 m to 90 m with bilinear. What goes wrong, and which methods should you use in rasterio, terra, and ArcPy?
5. Why did tonight's notebook reproject the watersheds to EPSG:32610 instead of warping the rasters to EPSG:2926?
6. `zonal_stats(..., categorical=True)` and `exact_extract(..., "frac")` return something different from `"mean"`. What, and why is the mean of a land cover raster meaningless?
7. Why should you reclassify a DEM before vectorizing it?
8. In xarray, what shape does `ndvi.mean(dim="time")` return, and what about `ndvi.mean(dim=["y", "x"])`? What question does each answer?

---
## What's Next — Module 7: Visualizing Spatial Data

Publication-quality maps with `matplotlib`, `ggplot2`, and `tmap`, plus interactive web maps — starting from the watershed summary and the suitability raster you just built.

**Stuck?** Post in Ed Discussion with the tag `#module6`. Include the error message, the cell that failed, and your environment (JupyterHub, ArcGIS Pro, or local).